<a href="https://colab.research.google.com/github/jiedeng99/ml/blob/main/projects/self_attention/notebooks/SelfAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
from google.colab import drive

try:
    # Mount Google Drive - Watch for a browser pop-up to grant access
    print("Attempting to mount Google Drive...")
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted successfully!")

    # Path to store the dataset in Drive
    drive_path = './Dataset.zip'

    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)

    # Only download if it doesn't exist in Drive
    if not os.path.exists(drive_path):
        print("Dataset not found in Drive. Downloading...")
        !gdown '1gaFy8RaQVUEXo2n0peCBR5gYKCB-mNHc' --output {drive_path}

    # Unzip to local runtime for faster access
    if not os.path.exists('./Dataset'):
        print("Unzipping dataset to local runtime...")
        !unzip -q {drive_path} -d ./
        print("Unzip complete.")
    else:
        print("Dataset already exists in local runtime.")

except Exception as e:
    print(f"Mounting failed: {e}")
    print("Please ensure you click 'Connect to Google Drive' in the pop-up and approve permissions.")

Attempting to mount Google Drive...
Mounted at /content/drive
Drive mounted successfully!
Unzipping dataset to local runtime...
Unzip complete.


In [11]:
import os
import json
import math
import csv
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.optim import Optimizer, AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm import tqdm

In [12]:
class myDataset(Dataset):
  def __init__(self, data_dir, segment_len=128):
    self.data_dir = data_dir
    self.segment_len = segment_len

    # Load the mapping from speaker neme to their corresponding id.
    mapping_path = Path(data_dir) / "mapping.json"
    mapping = json.load(mapping_path.open())
    self.speaker2id = mapping["speaker2id"]

    metadata_path = Path(data_dir) / "metadata.json"
    metadata = json.load(open(metadata_path))["speakers"]

    self.speaker_num = len(metadata.keys())
    self.data = []
    for speaker in metadata.keys():
      for utterances in metadata[speaker]:
        self.data.append([utterances["feature_path"], self.speaker2id[speaker]])

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    feat_path, speaker = self.data[index]
    mel = torch.load(os.path.join(self.data_dir, feat_path))

    if len(mel) > self.segment_len:
      start = random.randint(0, len(mel) - self.segment_len)
      mel = torch.FloatTensor(mel[start:start+self.segment_len])
    else:
      mel = torch.FloatTensor(mel)
    speaker = torch.FloatTensor([speaker]).long()
    return mel, speaker

  def get_speaker_number(self):
    return self.speaker_num

In [13]:
def collate_batch(batch):
  mel, speaker = zip(*batch)
  mel = pad_sequence(mel, batch_first=True, padding_value=-20)
  return mel, torch.FloatTensor(speaker).long()

def get_dataloader(data_dir, batch_size, n_workers):
  dataset = myDataset(data_dir)
  speaker_num = dataset.get_speaker_number()
  trainlen = int(0.9 * len(dataset))
  lengths = [trainlen, len(dataset) - trainlen]
  trainset, validset = random_split(dataset, lengths)

  train_loader = DataLoader(
      trainset,
      batch_size=batch_size,
      shuffle=True,
      drop_last=True,
      num_workers=n_workers,
      pin_memory=True,
      collate_fn=collate_batch,
  )

  valid_loader = DataLoader(
      validset,
      batch_size=batch_size,
      drop_last=True,
      num_workers=n_workers,
      pin_memory=True,
      collate_fn=collate_batch,
  )

  return train_loader, valid_loader, speaker_num

In [14]:
class Classifier(nn.Module):
  def __init__(self, d_model=80, n_spks=600, dropout=0.1):
    super().__init__()
    self.prenet = nn.Linear(40, d_model)  # Why do we have the number 40 here???
    self.encoder_layer = nn.TransformerEncoderLayer(
        d_model=d_model, dim_feedforward=256, nhead=2
    )
    self.pred_layer = nn.Sequential(
        nn.Linear(d_model, d_model),
        nn.ReLU(),
        nn.Linear(d_model, n_spks),
    )

  def forward(self, mels):
    out = self.prenet(mels)
    out = out.permute(1, 0 , 2)
    out = self.encoder_layer(out)
    out = out.transpose(0, 1)
    stats = out.mean(dim=1)
    out = self.pred_layer(stats)
    return out

In [15]:
def get_cosine_schedule_with_warmup(
    optimizer: Optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
    num_cycles: float = 0.5,
    last_epoch: int = -1,
):

  def lr_lambda(current_step):
    if current_step < num_warmup_steps:
      return float(current_step) / float(max(1, num_warmup_steps))
    progress = float(current_step - num_warmup_steps) / float(
        max(1, num_training_steps - num_warmup_steps))
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress)))

  return LambdaLR(optimizer, lr_lambda, last_epoch)

In [16]:
def model_fn(batch, model, criterion, device):
  mels, labels = batch
  mels, labels = mels.to(device), labels.to(device)

  outs = model(mels)

  loss = criterion(outs, labels)

  preds = outs.argmax(1)

  accuracy = torch.mean((preds == labels).float())

  return loss, accuracy

In [17]:
def validate(dataloader, model, criterion, device):
  model.eval()
  running_loss = 0.0
  running_accuracy = 0.0
  pbar = tqdm(total=len(dataloader.dataset), ncols = 0, desc="Valid", unit=" uttr")
  for i, batch in enumerate(dataloader):
    with torch.no_grad():
      loss, accuracy = model_fn(batch, model, criterion, device)
      running_loss += loss.item()
      running_accuracy += accuracy.item()

    pbar.update(dataloader.batch_size)
    pbar.set_postfix(
      loss=f"{running_loss / (i + 1):.2f}",
      accuracy=f"{running_accuracy / (i + 1):.2f}",
    )
  pbar.close()
  model.train()
  return running_accuracy / len(dataloader)

In [18]:
def parse_args():
  config = {
      "data_dir": "./Dataset",
      "save_path": "./model.ckpt",
      "batch_size": 32,
      "n_workers": 8,
      "valid_steps": 2000,
      "warmup_steps": 1000,
      "save_steps": 4000, #10000,
      "total_steps": 5000, #70000,
  }
  return config

def main(
    data_dir,
    save_path,
    batch_size,
    n_workers,
    valid_steps,
    warmup_steps,
    total_steps,
    save_steps,
):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"[Info]: Use {device}")
  train_loader, valid_loader, speaker_num = get_dataloader(data_dir, batch_size, n_workers)
  train_iterator = iter(train_loader)
  print("[Info]: Dataloader Done", flush=True)

  model = Classifier(n_spks=speaker_num).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = AdamW(model.parameters(), lr=1e-3)
  scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
  print("[Info]: Model, Criterion, Optimizer, Scheduler Done", flush=True)

  best_accuracy = -1.0
  best_state_dict = None

  pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

  for step in range(total_steps):
    try:
      batch = next(train_iterator)
    except StopIteration:
      train_iterator = iter(train_loader)
      batch = next(train_iterator)

    loss, accuracy = model_fn(batch, model, criterion, device)
    batch_loss = loss.item()
    batch_accuracy = accuracy.item()

    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    pbar.update()
    pbar.set_postfix(
      loss=f"{batch_loss:.2f}",
      accuracy=f"{batch_accuracy:.2f}",
      step=step + 1,
    )

    if (step + 1) % valid_steps == 0:
      pbar.close()
      valid_accuracy = validate(valid_loader, model, criterion, device)
      if valid_accuracy > best_accuracy:
        best_accuracy = valid_accuracy
        best_state_dict = model.state_dict()

      pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

    if (step + 1) % save_steps == 0 and best_state_dict is not None:
      torch.save(best_state_dict, save_path)
      pbar.write(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")
  pbar.close()

if __name__ == "__main__":
  main(**parse_args())

[Info]: Use cuda
[Info]: Dataloader Done
[Info]: Model, Criterion, Optimizer, Scheduler Done


Train: 100% 2000/2000 [00:14<00:00, 139.84 step/s, accuracy=0.31, loss=3.45, step=2000]
Valid: 100% 6944/6944 [00:01<00:00, 6538.21 uttr/s, accuracy=0.21, loss=3.84]
Train: 100% 2000/2000 [00:13<00:00, 150.46 step/s, accuracy=0.31, loss=3.29, step=4000]
Valid: 100% 6944/6944 [00:01<00:00, 6743.73 uttr/s, accuracy=0.31, loss=3.16]
Train:   2% 31/2000 [00:00<00:13, 147.87 step/s, accuracy=0.50, loss=2.25, step=4031]

Step 4000, best model saved. (accuracy=0.3132)


Train:  50% 1000/2000 [00:06<00:06, 153.38 step/s, accuracy=0.22, loss=3.11, step=5000]


In [21]:
class InferenceDataset(Dataset):
  def __init__(self, data_dir):
    testdata_path = Path(data_dir) / "testdata.json"
    metadata = json.load(testdata_path.open())
    self.data_dir = data_dir
    self.data = metadata["utterances"]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    utterance = self.data[index]
    feat_path = utterance["feature_path"]
    mel = torch.load(os.path.join(self.data_dir, feat_path))
    return feat_path, mel

def inference_collate_batch(batch):
  feat_paths, mels = zip(*batch)

  return feat_paths, torch.stack(mels)

In [23]:
import csv

def parse_args():
  config = {
      "data_dir": "./Dataset",
      "model_path": "./model.ckpt",
      "output_path": "./output.csv",
  }
  return config

def main(data_dir, model_path, output_path):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"[Info]: Use {device}")

  mapping_path = Path(data_dir) / "mapping.json"
  mapping = json.load(mapping_path.open())

  dataset = InferenceDataset(data_dir)
  dataloader = DataLoader(
      dataset,
      batch_size=1,
      shuffle=False,
      drop_last=False,
      num_workers=8,
      # pin_memory=False,
      collate_fn=inference_collate_batch,
  )

  print(f"[Info]: Finish loading data", flush=True)

  speaker_num = len(mapping["id2speaker"])
  model = Classifier(n_spks = speaker_num).to(device)
  model.load_state_dict(torch.load(model_path))
  model.eval()
  print(f"[Info]: Finish loading model", flush=True)

  results = [["id", "Category"]]
  for feat_paths, mels in tqdm(dataloader):
    with torch.no_grad():
      mels = mels.to(device)
      outs = model(mels)
      preds = outs.argmax(1).cpu().numpy()
      for feat_path, pred in zip(feat_paths, preds):
        results.append([feat_path, mapping["id2speaker"][str(pred)]])

  with open(output_path, "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerows(results)

if __name__ == "__main__":
  main(**parse_args())

[Info]: Use cuda
[Info]: Finish loading data
[Info]: Finish loading model


100%|██████████| 6000/6000 [00:10<00:00, 574.58it/s]
